## 1. 输出解析器 OutPutParser

### 1.1 通过Prompt约束（JsonOutputParser）

In [ ]:
# 1.导入相关依赖
import os, json
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
# 2. 实例化模型
llm = ChatOpenAI(
    model="gpt-4o",
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY")
)


class Prime(BaseModel):
    prime: list[int] = Field(description="素数")  # 必填字段    类型 list[int]
    count: list[int] = Field(description="小于该素数的素数个数")  # 必填字段   类型 list[int]


# 3. 实例化输出解析器
json_out_put_parser = JsonOutputParser(pydantic_object=Prime)

# 4. 调用模型(json格式字符串)
reps = llm.invoke([
    ("system", json_out_put_parser.get_format_instructions()),
    ("user", "任意生成5个1000-100000之间的素数，并标出小于该素数的素数个数")

])
### json格式的字符串（json字符串转成json对象 不是的，返回的只是一个json格式的字符串而已）

json_object = json_out_put_parser.invoke(input=reps.content)

print(type(json_object))
print(json_object)

# 使用输出解析器主要是使用两个能力
# 1. 用结构化指令
# 2. 用parse方法

In [ ]:
from pydantic import BaseModel, Field


# 定义一个数据模型
class User(BaseModel):
    name: str  # 名字必须是字符串
    age: int  # 年龄必须是整数
    email: str = Field(description="用户的邮箱地址")  # 必填字段

# 创建实例 - 数据会自动校验和转换
# user = User(name="张三", age=25, email="zhangsan@example.com")
# print(user.name)   # 张三
# print(user.age)    # 25

# # 类型不匹配时，Pydantic会自动尝试转换
# user2 = User(name="李四", age="30",email="xxxxx")  # age是字符串"30"，会自动转为整数30
# print(user2.age)   # 30 (已自动转换为int)
# #
# # 数据不合法时会抛出错误
# # User(name="王五", age="abc")  # 报错：无法将"abc"转为整数

### 1.2 通过厂商原生能力

### 1.2.1 OpenAI

In [ ]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()


class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."}
    ],
    response_format=CalendarEvent
)

print(response.choices[0].message.parsed)
# CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [ ]:
from google import genai
from pydantic import BaseModel


class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


client = genai.Client(
    api_key="sk-OdqRypFlfJLKYvLmV6GsG9j0u6CRFBYKErn4xV1Wm0R3q0y9",
    http_options={
        "base_url": "https://api.openai-proxy.org/google"
    }
)

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Alice and Bob are going to a science fair on Friday.",
    config={
        "response_mime_type": "application/json",
        "response_json_schema": CalendarEvent.model_json_schema(),
    },
)
print(response.text)
event = CalendarEvent.model_validate_json(response.text)
print(type(event))
print(type(event))

### 1.3  LangChain统一封装 with_structured_output

#### 1.4 PydanticOutputParser 输出解析器

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, field_validator
from langchain_openai import ChatOpenAI
from dotenv import  load_dotenv
load_dotenv()

class MovieReview(BaseModel):
    """电影评论结构"""
    title: str = Field(description="电影标题")
    rating: int = Field(description="评分，1-10分", ge=1, le=10)
    summary: str = Field(description="剧情简介")
    recommended: bool = Field(description="是否推荐")

    @field_validator('rating')
    @classmethod
    def rating_must_be_valid(cls, v):
        if v < 1 or v > 10:
            raise ValueError('评分必须在1-10之间')
        return v


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = PydanticOutputParser(pydantic_object=MovieReview)

# 将格式说明注入到Prompt中
prompt = ChatPromptTemplate.from_messages([
    ("system", "{instruction}"),
    ("human", "评价电影《盗梦空间》")
])

chain = prompt | llm | parser
result = chain.invoke({"instruction": parser.get_format_instructions()})
print(result)
# print(f"电影: {result.title}, 评分: {result.rating}/10")

#### 1.5 自定义输出解析器

In [4]:
from langchain_core.output_parsers import BaseOutputParser

class CommaListParser(BaseOutputParser):
    """将逗号分隔的文本解析为列表"""

    def parse(self, text: str):
        # 去除空白后按逗号分割
        return [item.strip() for item in text.split(",")]

# 使用
parser = CommaListParser()
result = parser.parse("苹果, 香蕉, 橘子")
print(result)  # ['苹果', '香蕉', '橘子']

['苹果', '香蕉', '橘子']
